# <font color="#418FDE" size="6.5" uppercase>**Filters Wrappers**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply variance and univariate filter methods to select informative features. 
- Choose classification or regression feature scores for the target type. 
- Use recursive feature elimination and cross-validated RFE. 


## **1. Filter Feature Selection**

### **1.1. Variance Threshold**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_01_01.jpg?v=1784016047" width="250">



>* Remove features with little variation
>* Fast first step before deeper selection

>* Works without a target variable
>* Keep rare important features with domain judgment

>* Choose thresholds carefully based on feature scale
>* Use as an initial noise filter



In [ ]:
#@title Python Code - Variance Threshold

# Demonstrate variance threshold feature selection.
# Low-variance columns carry little information.
# The output shows removed features.

import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# Build a tiny dataset with varied feature behavior.
data = pd.DataFrame(
    {
        "constant_id": [1, 1, 1, 1, 1, 1],
        "almost_always_zero": [0, 0, 0, 0, 0, 1],
        "age_years": [22, 35, 41, 29, 58, 46],
        "monthly_visits": [2, 5, 3, 8, 1, 6],
    }
)

# Check that the example has the expected shape.
if data.shape != (6, 4):
    raise ValueError("Expected 6 rows and 4 features.")

# VarianceThreshold keeps columns above the chosen variance.
selector = VarianceThreshold(threshold=0.20)
selected_array = selector.fit_transform(data)

# Convert the selected array back to labeled columns.
kept_features = selector.get_feature_names_out(data.columns)
selected_data = pd.DataFrame(selected_array, columns=kept_features)

# Show each feature variance before filtering.
variances = data.var(axis=0, ddof=0).round(3)
print("Feature variances:")
print(variances.to_string())

# Show which features survived the threshold.
print("Kept features:", list(kept_features))
print("Original shape:", data.shape)
print("Selected shape:", selected_data.shape)



### **1.2. Univariate Feature Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_01_02.jpg?v=1784016045" width="250">



>* Scores each feature separately against the target
>* Fast, interpretable screening for informative predictors

>* Match scoring method to target type
>* Rank features using appropriate statistical evidence

>* Reduces noise and simplifies high-dimensional datasets
>* May miss interactions or keep redundant features



In [ ]:
#@title Python Code - Univariate Feature Selection

# This example ranks features using univariate scores.
# Each feature is tested separately against the target.
# The plot shows which features are selected.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn import __version__ as sklearn_version

# Create a small classification dataset with known informative features.
feature_names = ["signal_1", "signal_2", "signal_3", "noise_1", "noise_2", "noise_3"]
X, y = make_classification(
    n_samples=200,
    n_features=6,
    n_informative=3,
    n_redundant=0,
    n_repeated=0,
    random_state=42,
    shuffle=False,
)

# Validate the basic shape before scoring features.
if X.shape != (200, 6) or y.shape[0] != 200:
    raise ValueError("Expected 200 rows and 6 feature columns.")

# Select the three strongest individual features.
selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X, y)
selected_mask = selector.get_support()
selected_names = np.array(feature_names)[selected_mask]

# Print a concise summary of the selected features.
print(f"scikit-learn version: {sklearn_version}")
print("Selected features: " + ", ".join(selected_names))
print("Higher F-scores mean stronger individual class separation.")

# Plot every feature score to compare selected and rejected columns.
scores = selector.scores_
colors = np.where(selected_mask, "tab:green", "tab:gray")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(feature_names, scores, color=colors)

# Label the chart for beginner interpretation.
ax.set_title("Univariate Feature Scores with SelectKBest")
ax.set_xlabel("Feature")
ax.set_ylabel("ANOVA F-score")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()



### **1.3. Percentile Feature Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_01_03.jpg?v=1784016049" width="250">



>* Keep a chosen percentage of top features
>* Reduce large datasets with scalable filtering

>* Rank features using target-matched statistical scores
>* Fast individually, but misses feature interactions

>* Balance feature reduction with retained information
>* Tune percentiles using training validation only



In [ ]:
#@title Python Code - Percentile Feature Selection

# Demonstrate percentile feature selection on classification data.
# SelectKBest keeps a chosen percentage of features.
# Printed scores show which features remain.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import f_classif

# Create data with informative and noisy features.
features, target = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=3,
    n_redundant=0,
    random_state=42,
)

# Name features so selected columns are easy to read.
feature_names = np.array(["feature_" + str(number) for number in range(10)])

# Keep the top thirty percent by univariate classification score.
selector = SelectPercentile(score_func=f_classif, percentile=30)
selected_features = selector.fit_transform(features, target)

# Validate that thirty percent of ten features gives three columns.
expected_columns = 3
actual_columns = selected_features.shape[1]

if actual_columns != expected_columns:
    raise ValueError("Unexpected number of selected features.")

# Read the selected feature names and their rounded scores.
selected_mask = selector.get_support()
selected_names = feature_names[selected_mask]
selected_scores = np.round(selector.scores_[selected_mask], 2)

print("scikit-learn version:", sklearn.__version__)
print("Original feature count:", features.shape[1])
print("Selected feature count:", actual_columns)
print("Selected feature names:", list(selected_names))
print("Selected feature scores:", list(selected_scores))



## **2. Score Functions**

### **2.1. Selecting Score Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_02_01.jpg?v=1784016060" width="250">



>* Match score functions to target type
>* Wrong scores can mislead feature rankings

>* Classification scores compare features across categories
>* Regression scores link features to numeric changes

>* Match scores to feature types and assumptions
>* Use filters before deeper model-based selection



In [ ]:
#@title Python Code - Selecting Score Functions

# This example compares score functions for target types.
# Classification scores fit categories, regression scores fit numbers.
# The output shows different appropriate feature rankings.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.datasets import make_regression
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import f_regression

print("scikit-learn version:", sklearn.__version__)

# Build one categorical-target dataset and one numeric-target dataset.
X_class, y_class = make_classification(
    n_samples=120,
    n_features=5,
    n_informative=2,
    n_redundant=0,
    random_state=42,
)

X_reg, y_reg = make_regression(
    n_samples=120,
    n_features=5,
    n_informative=2,
    noise=10.0,
    random_state=42,
)

# Validate that both examples have matching rows and features.
if X_class.shape != X_reg.shape:
    raise ValueError("Both examples should have the same feature shape.")

if len(y_class) != X_class.shape[0] or len(y_reg) != X_reg.shape[0]:
    raise ValueError("Each target must match the number of rows.")

# Use a classification score for the categorical target.
class_scores, class_p_values = f_classif(X_class, y_class)
class_order = np.argsort(class_scores)[::-1]

# Use a regression score for the continuous target.
reg_scores, reg_p_values = f_regression(X_reg, y_reg)
reg_order = np.argsort(reg_scores)[::-1]

feature_names = np.array(["feature_0", "feature_1", "feature_2", "feature_3", "feature_4"])

print("Classification target: use f_classif for category separation.")
print("Top classification features:", list(feature_names[class_order[:3]]))
print("Classification scores:", np.round(class_scores[class_order[:3]], 2).tolist())
print("Regression target: use f_regression for numeric association.")
print("Top regression features:", list(feature_names[reg_order[:3]]))
print("Regression scores:", np.round(reg_scores[reg_order[:3]], 2).tolist())
print("Lesson: match the score function to the target type.")



### **2.2. Classification Feature Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_02_02.jpg?v=1784016056" width="250">



>* Use scores for categorical target labels
>* Higher scores indicate stronger class separation

>* Match scores to feature and target types
>* Different scores detect different class relationships

>* High scores suggest usefulness, not guaranteed improvement
>* Validate features within a careful workflow



In [ ]:
#@title Python Code - Classification Feature Scores

# Compare classification feature score functions.
# Match each score to feature type.
# Higher scores suggest stronger class association.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import chi2
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import KBinsDiscretizer

# Create a small classification dataset with known informative features.
features, target = make_classification(
    n_samples=300,
    n_features=5,
    n_informative=2,
    n_redundant=0,
    n_repeated=0,
    random_state=42,
)

# Validate the basic shape before scoring features.
if features.shape != (300, 5):
    raise ValueError("Expected 300 rows and 5 feature columns.")

# Name features so the score table is easy to read.
feature_names = ["feature_0", "feature_1", "feature_2", "feature_3", "feature_4"]
feature_table = pd.DataFrame(features, columns=feature_names)

# ANOVA F scores work with numeric features and class labels.
f_scores, f_p_values = f_classif(feature_table, target)

# Chi-square needs nonnegative count-like or binned features.
binner = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
binned_features = binner.fit_transform(feature_table)
chi2_scores, chi2_p_values = chi2(binned_features, target)

# Mutual information can capture more general dependence.
mi_scores = mutual_info_classif(feature_table, target, random_state=42)

# Put the three classification scores side by side.
score_table = pd.DataFrame(
    {
        "ANOVA_F": np.round(f_scores, 2),
        "Chi2_binned": np.round(chi2_scores, 2),
        "Mutual_info": np.round(mi_scores, 2),
    },
    index=feature_names,
)

# Sort by one score to highlight the strongest candidates.
score_table = score_table.sort_values("ANOVA_F", ascending=False)

print("scikit-learn version:", sklearn.__version__)
print("Classification target classes:", sorted(np.unique(target).tolist()))
print("Top feature scores for a categorical target:")
print(score_table.head(5).to_string())



### **2.3. Regression Feature Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_02_03.jpg?v=1784016058" width="250">



>* Use regression scores for continuous targets
>* Select features linked to numeric outcomes

>* Linear scores capture straight-line target relationships
>* Other scores detect curved or complex patterns

>* Scores judge features separately, missing interactions
>* Validate selected features in full workflows



In [ ]:
#@title Python Code - Regression Feature Scores

# This example compares regression feature score functions.
# Linear and nonlinear relationships can score differently.
# The printed ranking shows which features matter.

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import mutual_info_regression

# Create a small deterministic regression dataset.
rng = np.random.default_rng(42)
sample_count = 200
linear_feature = rng.normal(size=sample_count)

# Add one curved feature and one mostly useless feature.
curved_feature = rng.uniform(-3, 3, size=sample_count)
noise_feature = rng.normal(size=sample_count)
noise = rng.normal(scale=0.4, size=sample_count)

# Build a continuous target from linear and curved patterns.
target = 3 * linear_feature + 2 * curved_feature**2 + noise
features = pd.DataFrame(
    {
        "linear_feature": linear_feature,
        "curved_feature": curved_feature,
        "noise_feature": noise_feature,
    }
)

# Validate the basic shape before scoring features.
if features.shape != (sample_count, 3):
    raise ValueError("Expected 200 rows and 3 feature columns.")

# f_regression looks for linear relationships with the target.
f_scores, p_values = f_regression(features, target)
mi_scores = mutual_info_regression(features, target, random_state=42)

# Put both regression score types side by side.
score_table = pd.DataFrame(
    {
        "f_regression": np.round(f_scores, 2),
        "mutual_info": np.round(mi_scores, 2),
    },
    index=features.columns,
)

# Sort by mutual information to highlight general dependence.
score_table = score_table.sort_values("mutual_info", ascending=False)
print("scikit-learn version:", sklearn.__version__)
print("Regression feature scores for a continuous target:")
print(score_table)



## **3. Recursive Feature Elimination**

### **3.1. RFE Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_03_01.jpg?v=1784016051" width="250">



>* RFE repeatedly removes least useful features
>* Model context guides compact feature selection

>* Model importance guides feature removal
>* Retraining updates rankings after each removal

>* Balance removal speed with feature selection care
>* Validate RFE to avoid overfitting



In [ ]:
#@title Python Code - RFE Basics

# This example demonstrates basic recursive feature elimination.
# RFE repeatedly removes weak features using a model.
# The output shows selected features and test accuracy.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

# Check that features and labels match correctly.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split before fitting RFE to avoid test data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Logistic regression supplies coefficients for RFE ranking.
base_model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)
rfe = RFE(estimator=base_model, n_features_to_select=5, step=2)

# Scaling is fitted only on training data inside the pipeline.
model = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", rfe),
    ("classifier", base_model),
])

# Fit the pipeline and evaluate the selected feature subset.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Read the RFE mask after training.
selected_mask = model.named_steps["rfe"].support_
selected_features = feature_names[selected_mask]

print("scikit-learn version:", sklearn.__version__)
print("Original feature count:", X.shape[1])
print("Selected feature count:", len(selected_features))
print("Selected features:", ", ".join(selected_features))
print("Test accuracy with selected features:", round(accuracy, 3))



### **3.2. Cross Validated RFE**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_03_02.jpg?v=1784016053" width="250">



>* Uses validation to choose feature count
>* Keeps features that generalize reliably

>* Link feature choices to validation performance
>* Identify simpler models or vital features

>* Best features depend on model and metric
>* Higher cost supports simpler reliable models



In [ ]:
#@title Python Code - Cross Validated RFE

# Demonstrate cross validated recursive feature elimination.
# RFECV chooses feature count using validation scores.
# The plot shows accuracy versus kept features.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

# Create data with informative and noisy features.
features, target = make_classification(
    n_samples=500,
    n_features=12,
    n_informative=4,
    n_redundant=2,
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape != (500, 12):
    raise ValueError("Expected 500 rows and 12 features.")

# Use a simple linear classifier for feature ranking.
classifier = LogisticRegression(max_iter=1000, random_state=42)
selector = RFECV(
    estimator=classifier,
    step=1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
)

# Scale features inside the pipeline before RFECV fits models.
model = make_pipeline(StandardScaler(), selector)
model.fit(features, target)

# Read the fitted RFECV step from the pipeline.
fitted_selector = model.named_steps["rfecv"]
kept_count = fitted_selector.n_features_
kept_names = np.array(["f" + str(i) for i in range(features.shape[1])])

# Prepare validation scores for each tested feature count.
mean_scores = fitted_selector.cv_results_["mean_test_score"]
feature_counts = np.arange(1, len(mean_scores) + 1)

print("scikit-learn version:", sklearn.__version__)
print("RFECV selected features:", kept_count)
print("Selected feature names:", list(kept_names[fitted_selector.support_]))

# Plot how validation accuracy changes as features are kept.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(feature_counts, mean_scores, marker="o")
ax.axvline(kept_count, color="red", linestyle="--", label="RFECV choice")

ax.set_title("RFECV validation accuracy by feature count")
ax.set_xlabel("Number of kept features")
ax.set_ylabel("Mean cross-validated accuracy")
ax.legend()

plt.show()



### **3.3. Wrapper Cost**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_03_03.jpg?v=1784016055" width="250">



>* RFE retrains models after removing features
>* Many features make wrappers costly

>* Cross-validation makes RFE more reliable
>* Repeated model fitting greatly increases computation

>* Narrow features before applying RFE
>* Balance accuracy, clarity, and computing cost



# <font color="#418FDE" size="6.5" uppercase>**Filters Wrappers**</font>


In this lecture, you learned to:
- Apply variance and univariate filter methods to select informative features. 
- Choose classification or regression feature scores for the target type. 
- Use recursive feature elimination and cross-validated RFE. 

In the next Lecture (Lecture B), we will go over 'Embedded Sequential'